In [1]:
# Importation des bibliothèques nécessaires
import pandas as pd
import json
import os
import psycopg2
from psycopg2.extras import Json
from psycopg2.extras import execute_values
import math
from datetime import datetime


In [2]:
filename = "resultats_rpls_2024_v3.xlsx"

In [ ]:
def clean_json(obj):
    if isinstance(obj, dict):
        return {k: clean_json(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [clean_json(v) for v in obj]
    elif isinstance(obj, float):
        if math.isinf(obj) or math.isnan(obj):
            return None
        return obj
    elif isinstance(obj, str):
        if obj.upper() in ("INF", "NA", "NAN", ""):
            return None
        return obj
    else:
        return obj

def charger_sheet(file_path, sheet_name, header_row):
    try:
        # Charger une feuille <sheet_name> et utiliser les en-têtes situées à la ligne <header_row> (en l'occurence ligne 6 qui correspond à l'index 5 en Python)
        df = pd.read_excel(file_path, sheet_name=sheet_name, header=header_row)
        print(f"✔️ Fichier chargé avec succès depuis la feuille '{sheet_name}'.")
        json_data = df.to_dict(orient='records')
        print(f"✔️ {len(json_data)} lignes converties en JSON depuis la feuille '{sheet_name}'.")
    except Exception as e:
        print(f"❌ Erreur lors du chargement du fichier : {e}")
        raise
    return json_data

def create_bronze_table (conn, table_name):
    # Création de la table bronze si elle n'existe pas
    # colonne created_at pour stocker la date d'import

    try:
        cursor = conn.cursor()
        create_table_query = f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            id SERIAL PRIMARY KEY,
            data JSONB,
            created_at TIMESTAMPTZ
        );
        """
        cursor.execute(create_table_query)
        conn.commit()
        print(f"✔️ Table '{table_name}' prête.")
    except Exception as e:
        print(f"❌ Erreur lors de la création de la table : {e}")
        conn.rollback()
        raise

def import_data_to_bronze_table (conn, table_name, json_data, created_at):# Insertion des données JSON dans la table
    try:
        cursor = conn.cursor()
        # mode itératif, plus lent, debugage facile
        insert_query = f"INSERT INTO {table_name} (created_at, data) VALUES (%s, %s)"
        for record in json_data:
            #print("Inserting:", record)  
            cleaned_record = clean_json(record)
            #print("cleaned_record:", cleaned_record)  
            cursor.execute(insert_query, (created_at, Json(cleaned_record)))
            conn.commit()
        print(f"✔️ {len(json_data)} lignes insérées dans la table '{table_name}'.")
        
        # mode bulk insert, plus rapide
        #insert_query = f"INSERT INTO {table_name} (data) VALUES %s"
        #json_records = [(json.dumps(clean_json(record)),) for record in json_data]
        #execute_values(cursor, insert_query, json_records)
        #conn.commit()
        #print(f"{len(json_records)} lignes insérées dans la table '{table_name}'.")

    except Exception as e:
        print(f"❌ Erreur lors de l'insertion des données : {e}")
        conn.rollback()
        raise
    finally:
        cursor.close()

def get_connexion():
    try:
        conn = psycopg2.connect(
            dbname=os.getenv('PG_DB_NAME'),
            user=os.getenv('PG_DB_USER'),
            password=os.getenv('PG_DB_PWD'),
            host="localhost",
            port="5432"
        )
        cursor = conn.cursor()
        print("✔️ Connexion à PostgreSQL réussie.")
    except Exception as e:
        print(f"❌ Erreur de connexion à PostgreSQL : {e}")
        raise
    return conn

def get_data_file_path(file_name): 
    # Définir le répertoire data local pour ce notebook
    notebook_dir = os.path.abspath('')
    print(f"🧾 notebook_dir: {notebook_dir}")
    data_folder = os.path.join(os.path.dirname(notebook_dir), 'data/imports/')
    print(f"🧾 file_folder: {data_folder}")
    file_path = os.path.join(data_folder, file_name)
    print(f"🧾 file_path: {file_path}")
    # Vérifier si le fichier existe
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ Fichier introuvable : {file_path}")
    print(f"✔️ Fichier trouvé.")
    print(f"✔️ Emplacement: {file_path}")
    return file_path




In [10]:
created_at = datetime.now()

# Définir le répertoire data local pour ce notebook
print("\n🚀 Recherche du fichier data {filename}")
file_path = get_data_file_path(filename)

# Charger le fichier Excel dans un DataFrame pandas
print("\n🚀 Chargement du fichier Excel dans un DataFrame pandas")
json_data_region = charger_sheet(file_path, "REGION", 5)
json_data_departement = charger_sheet(file_path, "DEPARTEMENT", 5)
json_data_communes = charger_sheet(file_path, "COMMUNES", 5)
json_data_epci = charger_sheet(file_path, "EPCI", 5)

# Connexion à PostgreSQL
print("\n🚀 Connexion à PostgreSQL")
conn = get_connexion()

# Création de la table bronze si elle n'existe pas
print("\n🚀 Création de la table bronze")
bronze_table_name_region = "bronze.logement_rpls_region"
bronze_table_name_departement = "bronze.logement_rpls_departement"
bronze_table_name_communes = "bronze.logement_rpls_communes"
create_bronze_table (conn, bronze_table_name_region)
create_bronze_table (conn, bronze_table_name_departement)
create_bronze_table (conn, bronze_table_name_communes)

# Insertion des données JSON dans la table bronze
print("\n🚀 Insertion des données JSON dans la table bronze")
import_data_to_bronze_table (conn, bronze_table_name_region, json_data_region, created_at)
import_data_to_bronze_table (conn, bronze_table_name_departement, json_data_departement, created_at)
import_data_to_bronze_table (conn, bronze_table_name_communes, json_data_communes, created_at)

conn.close()

print("\n✅ Fin du traitement.")


🚀 Recherche du fichier data {filename}
🧾 notebook_dir: /workspaces/13_odis/notebooks
🧾 file_folder: /workspaces/13_odis/data/imports/
🧾 file_path: /workspaces/13_odis/data/imports/resultats_rpls_2024_v3.xlsx
✔️ Fichier trouvé.
✔️ Emplacement: /workspaces/13_odis/data/imports/resultats_rpls_2024_v3.xlsx

🚀 Chargement du fichier Excel dans un DataFrame pandas
✔️ Fichier chargé avec succès depuis la feuille 'REGION'.
✔️ 23 lignes converties en JSON depuis la feuille 'REGION'.
✔️ Fichier chargé avec succès depuis la feuille 'DEPARTEMENT'.
✔️ 96 lignes converties en JSON depuis la feuille 'DEPARTEMENT'.
✔️ Fichier chargé avec succès depuis la feuille 'COMMUNES'.
✔️ 16859 lignes converties en JSON depuis la feuille 'COMMUNES'.
✔️ Fichier chargé avec succès depuis la feuille 'EPCI'.
✔️ 1325 lignes converties en JSON depuis la feuille 'EPCI'.

🚀 Connexion à PostgreSQL
✔️ Connexion à PostgreSQL réussie.

🚀 Création de la table bronze
❌ Erreur lors de la création de la table : syntax error at o

SyntaxError: syntax error at or near "without"
LINE 5:             created_at TIMESTAMPTZ without time zone DEFAULT...
                                           ^
